# Muss Buss

http://muss-buss-env.2qxazt2xnm.us-east-1.elasticbeanstalk.com/

In [1]:
from requests import Session
import pandas as pdˇ
import time

In [688]:
class Card(object):
    
    def __init__(self, **kwargs):
        self.bust = kwargs.get('bust')
        self.fill = kwargs.get('fill')
        self.type = kwargs.get('type')
        self.rolls = kwargs.get('rolls')
        self.score = kwargs.get('score')
        
    def __repr__(self):
        return f'<{self.type}>'


class Game(object):

    def __init__(self, **kwargs):
        self.game_number = kwargs.get('game_number')
        self.players = kwargs.get('players'),
        self.deck_size= kwargs.get('deckSize'),
        self.current_player = kwargs.get('currentPlayer'),
        self.turn = kwargs.get('turn'),
        self.score = kwargs.get('score'),
        self.card = kwargs.get('card'),
        # self.cards = [Card(**card) for card in self.card]
        self.dice = kwargs.get('dice'),
        self.invalid_actions = kwargs.get('invalidActions')
        
    def pprint(self):
        '''Pretty print the current game information.'''
        players = []
        for player in self.players[0]:
            if player['name'] == self.current_player[0]['name']:
                players.append(f'***{player["name"]} ({player["score"]})***')
            else:
                players.append(f'{player["name"]} ({player["score"]})')
        players = ", ".join(players)
        card = f'Card: {self.card[0]["type"]}' if self.card[0] else 'Card:'
        dice = f'Dice: [{", ".join(str(dice["value"]) for dice in self.dice[0])}]' if self.dice[0] else 'Dice:'
        holds = []
        if self.dice[0]:
            for die in self.dice[0]:
                if not die['live']:
                    holds.append('X')
                    continue
                holds.append('H' if die['held'] else ' ')
        holds = f'       ' + "  ".join(holds)
        
        m = f'Game #{self.game_number}\n{players}\n  {card}\n  {dice}\n  {holds}'
        print(m)
        return
        
    def __repr__(self):
        return f'GAME: {", ".join([player["name"] for player in self.players[0]])}'


class Client(object):
    
    def __init__(self, username, password, wait=0.3, game_number=None, verbose=True):
        self._wait = wait
        self.verbose = verbose
        self.session = Session()
        self.host = 'http://muss-buss-env.2qxazt2xnm.us-east-1.elasticbeanstalk.com'
        r = self.__authorize__(username, password)
        self._sid = r.cookies['connect.sid']
        self.headers = {'Connection': 'keep-alive',
                        'Accept': 'application/json, text/plain, */*',
                        'User-Agent': 'gurk',
                        'Accept-Encoding': 'gzip, deflate',
                        'Accept-Language': 'en-US,en;q=0.9,fr;q=0.8',
                        'Cookie': f'connect.sid={self._sid}'}
        data = r.json()        
        self.id = data['id']
        self.email = data['email']
        self.name = data['name']
        self.games = data['games']
        self.game_number = game_number if game_number else self.games[0]['id']
        self.root = f"{self.host}/api/games/{self.game_number}"
        self.update_game()
        
    def send_request(self, query, method='get'):
        url = f'{self.root}/{query}'
        r = self.session.request(method=method,
                                 url=url,
                                 headers=self.headers)
        self.update_game(r)
        return r

    def __authorize__(self, username, password):
        headers = {'content type': 'application/json'}
        url = f'{self.host}/auth/login'
        auth = {'email': username, 'password': password}
        return self.session.post(url, headers=headers, data=auth)

    def pprint(self):
        self.game.pprint()

    def update_game(self, r=None):
        if not r:
            r = self.session.get(self.root)
        self.game = Game(**r.json(), game_number=self.game_number)
        if self.verbose:
            self.pprint()

    def draw(self):
        q = "draw"
        return self.send_request(q)

    def roll(self):
        q = "roll"
        return self.send_request(q)

    def pass_(self):
        q = "pass"
        return self.send_request(q)

    def stop(self):
        q = "stop"
        return self.send_request(q)
    
    def hold(self, dice_id=6):
        q = f"hold?holdId={dice_id}"
        return self.send_request(q)
    
    def __repr__(self):
        self.update_game()
        return f'({self.name}) {self.game}'

    def __del__(self):
        self.session.close()

In [692]:
class Player(Client):
    
    def __init__(self, *args, **kwargs):
        super(Player, self).__init__(*args, **kwargs)

    def take_turn(self):
        player = self.game.current_player[0]
        print(f"Current player: {player['name']}")
        print(f"Starting score: {player['score']}")
        actions = self.game.invalid_actions
        if not actions['invalidStop']:
            print('Stopping...')
            self.stop()
            return
        if not actions['invalidDraw']:
            print('Drawing a card...')
            self.draw()
            return
        if not actions['invalidRoll']:
            print('Rolling...')
            self.roll()
            return
        if not actions['invalidHolds']:
            print("Holding...")
            self.hold()
            return

In [715]:
del(p1, p2)
p1 = Player(username='john', password='123', game_number=14, verbose=False)
p2 = Player(username='john.w.millr@gmail.com', password='<redacted>', game_number=14, verbose=False)

In [722]:
p2.pprint()

Game #14
john (6850), ***john.w.millr@gmail.com (6050)***
  Card: bonus400
  Dice:
         


0

### Play a game

In [712]:
n = 1
N = 5000
while not any([p1.game.players[0][0]['winner'], p2.game.players[0][-1]['winner']]):
#     p1.update_game()
    p2.update_game()

#     if p1.game.current_player[0]['name'] == p1.name:
#         p1.take_turn()
#         if n % N == 0 or n == 1:
#             print(f'Turn #{n}')
#             p1.pprint()
#         continue
    if p2.game.current_player[0]['name'] == p2.name:
        p2.take_turn()
        if n % N == 0 or n == 1 or True:
            print(f'Turn #{n}')
            p2.pprint()
    time.sleep(0.5)
    n +=1
print('Winner!!')

Current player: john.w.millr@gmail.com
Starting score: 5300
Drawing a card...
Turn #24

PLAYER 2's TURN
Game #14
john (3450), ***john.w.millr@gmail.com (5300)***
  Card: bonus500
  Dice:
         
Current player: john.w.millr@gmail.com
Starting score: 5300
Rolling...
Turn #25

PLAYER 2's TURN
Game #14
john (3450), ***john.w.millr@gmail.com (5300)***
  Card: bonus500
  Dice: [2, 4, 6, 5, 6, 2]
                         
Current player: john.w.millr@gmail.com
Starting score: 5300
Holding...
Turn #26

PLAYER 2's TURN
Game #14
john (3450), ***john.w.millr@gmail.com (5300)***
  Card: bonus500
  Dice: [2, 4, 6, 5, 6, 2]
                  H      
Current player: john.w.millr@gmail.com
Starting score: 5300
Stopping...
Turn #27

PLAYER 2's TURN
Game #14
***john (3450)***, john.w.millr@gmail.com (5350)
  Card:
  Dice:
         
Current player: john.w.millr@gmail.com
Starting score: 5350
Drawing a card...
Turn #76

PLAYER 2's TURN
Game #14
john (3700), ***john.w.millr@gmail.com (5350)***
  Card: b

KeyboardInterrupt: 

In [699]:
p2.pprint()

Game #10
john (10150), ***john.w.millr@gmail.com (9750)***
  Card: bonus300
  Dice:
         


In [704]:
p2.game.invalid_actions

{'invalidDraw': True,
 'invalidRoll': True,
 'invalidStop': True,
 'invalidPass': True,
 'invalidHolds': True}

In [700]:
p2.game.invalid_actions

{'invalidDraw': True,
 'invalidRoll': False,
 'invalidStop': True,
 'invalidPass': True,
 'invalidHolds': True}

In [650]:
p1.game.players[0][-1].keys()

dict_keys(['id', 'name', 'score', 'turns', 'winner'])

In [639]:
p2.take_turn()

Current player: john
Starting score: 2650
Drawing a card...
Game #10
john (3350), ***john.w.millr@gmail.com (2000)***
  Card: fill1000
  Dice:
         


In [610]:
pd.Series(p1.game.invalid_actions)

invalidDraw      True
invalidRoll      True
invalidStop      True
invalidPass      True
invalidHolds    False
dtype: bool